In [1]:
from gensim.models import Word2Vec
import pandas as pd
import pickle
import numpy as np

with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/omim_id_name.dict', 'rb') as handle:
    omim_id_name = pickle.load(handle)
    
# with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/gene_id_name.dict', 'rb') as handle:
#     gene_id_name = pickle.load(handle)

with open('/home/peng/PycharmProjects/featurematch/data/processed/final/ids/hpo_id_name.dict', 'rb') as handle:
    hpo_id_name = pickle.load(handle)
    
    
training_patient_disease = {}
training_patient_disease_id = {}
training_disease_id_number = {}

training_patients = '/home/peng/PycharmProjects/featurematch/data/processed/patients/reform/training/all_training_patient.tsv'

df = pd.read_csv(training_patients, sep='\t', header=None)

for row in df.values.tolist():
        patient_id = row[0]
        disease_id = row[1]
        training_patient_disease_id[patient_id] = disease_id
        
        if disease_id in training_disease_id_number:
            training_disease_id_number[disease_id] += 1
        else:
            training_disease_id_number[disease_id] = 1
        
        if disease_id in omim_id_name:
            disease_name = omim_id_name[disease_id]
        training_patient_disease[patient_id] = disease_name
        
model = Word2Vec.load('../../../model/withoutgene_omim_with_patients/with_patients.model')

In [28]:
evaluation_save = list()
evaluation_vis = list()

evaluation_patients = '/home/peng/PycharmProjects/featurematch/data/processed/patients/reform/evaluation/evaluation_pedia.tsv'
with open(evaluation_patients, 'r') as e_file:
    content = e_file.read().splitlines()
    content = [x.split('\t') for x in content]
    for line in content:
        current_patient=[]
        feature_filtered_nodes= []     
        patient_id = line[0]
        disease_id = line[1]
        features = line[2].split(',')
        absent_features = line[4].split(',')
        if absent_features == ['unknown']:
            absent_features = []
            
        similar_nodes = model.most_similar(positive=features, negative=absent_features, topn=10000)
        
        for node in similar_nodes:
            if node[0].startswith('OMIM'):
                feature_filtered_nodes.append(node[0])
        rank = feature_filtered_nodes.index(disease_id) + 1
        evaluation_save.append([patient_id, disease_id, training_disease_id_number[disease_id], ','.join(features), str(len(features)), str(len(absent_features)), ','.join(feature_filtered_nodes[:10]), str(rank)])    
        evaluation_vis.append([patient_id, omim_id_name[disease_id], training_disease_id_number[disease_id], ','.join([hpo_id_name[feature] for feature in features]), len(features), len(absent_features), [omim_id_name[omim_id] for omim_id in feature_filtered_nodes[:10]], str(rank)])

/home/peng/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:18: DeprecationWarning: Call to deprecated `most_similar` (Method will be removed in 4.0.0, use self.wv.most_similar() instead).


In [29]:
pd.set_option('display.max_colwidth', -1)
saveframe = pd.DataFrame(evaluation_save, columns = ['patient_id', 'omim_id', 'no_patients', 'features', 'no_features','no_absfeatures', 'results_withp_withabs', 'rank_withp_withabs'])
saveframe.to_csv('../../../model/withoutgene_omim_with_patients/evaluation_with_absenthpo.tsv', sep='\t', index=None)

In [30]:
visframe = pd.DataFrame(evaluation_vis, columns = ['patient_id', 'omim_id', 'no_patients', 'features', 'no_features', 'no_absfeatures', 'results_withp_withabs', 'rank_withp_withabs'])
visframe.to_excel('../../../model/withoutgene_omim_with_patients/evaluation_with_absenthpo.xlsx', index=None)